In [0]:
# ------------------------------------------------------------
# INPUT PARAMETERS
# ------------------------------------------------------------

dbutils.widgets.text(
    "pipeline_run_id",
    ""
)

dbutils.widgets.text(
    "file_hash",
    ""
)

pipeline_run_id = dbutils.widgets.get("pipeline_run_id")
file_hash = dbutils.widgets.get("file_hash")

print(f"ADF Pipeline Run ID: {pipeline_run_id}")
print(f"File Hash: {file_hash}")

In [0]:
# ------------------------------------------------------------
# SILVER 1 — Read Bronze Batch
# ------------------------------------------------------------

from pyspark.sql.functions import col

bronze_path = (
    "abfss://bronze@olistdev1.dfs.core.windows.net/"
    "SAP/Customer"
)

df_bronze = (
    spark.read
    .format("delta")
    .load(bronze_path)
)

print(f"Total Bronze records: {df_bronze.count()}")

# ------------------------------------------------------------
# Validate input parameters
# ------------------------------------------------------------

if not file_hash:
    raise Exception(
        "SILVER_INPUT: file_hash parameter is required."
    )

if not pipeline_run_id:
    raise Exception(
        "SILVER_INPUT: pipeline_run_id parameter is required."
    )

print(f"Processing file hash: {file_hash}")
print(f"Processing pipeline run ID: {pipeline_run_id}")

# ------------------------------------------------------------
# Select exact Bronze batch
# ------------------------------------------------------------

df_bronze_batch = (
    df_bronze
    .filter(
        (col("file_hash") == file_hash)
        &
        (col("pipeline_run_id") == pipeline_run_id)
    )
)

bronze_batch_count = df_bronze_batch.count()

print(
    f"Bronze batch record count: "
    f"{bronze_batch_count}"
)

if bronze_batch_count == 0:
    raise Exception(
        "SILVER_BATCH: FAIL - "
        "No Bronze records found for the supplied "
        "file_hash and pipeline_run_id."
    )

print("SILVER_BATCH: SUCCESS")

display(df_bronze_batch.limit(10))